In [ ]:
import matplotlib.pyplot as plt
#import seaborn as sns
# Ensure plots render inline inside Jupyter
%matplotlib inline 

import json 

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

input_file = "phase5.json"
output_file = "portfolio_distribution.json"

In [ ]:
mc_results = None
with open(input_file, encoding="utf-8") as f:
    mc_results = json.load(f)
perf_data = mc_results["perf_data"]
sim_data = mc_results["simulations"]
models = [m for m in sim_data.keys()]

initial_nav = mc_results["initial_nav"]
years = mc_results["years"]
paths = mc_results["total_paths"]

run_stats = {}

def annualize(ret, years):
    neg = np.sign(ret)
    ret = abs(ret)
    a = np.pow(ret + 1, 1 / years) - 1
    return float(neg * a)

for m in models:
    r = sim_data[m]
    run_stats[m] = []
    for sim in r:
        yearly_spending = f"{sim["spending"]:,.0f}"
        equity = f"{sim["equity"]*100.0:02.0f}-{sim["ladder"]*100.0:02.0f}"
        df_data = pd.DataFrame(sim["results"])
        df_data["Returns"] = (df_data["Terminal NAV"] - initial_nav) / initial_nav
        ruin = len(df_data[df_data["Terminal NAV"] < 0])
        ret_mean = df_data["Returns"].mean()
        mdds_mean = df_data["Max Drawdowns"].mean()
        entry = {}
        entry["spending"] = yearly_spending
        entry["weight"] = equity
        entry["ruin_rate"] = ruin * 100.0 / paths

        entry["p1_return"] = df_data["Returns"].quantile(0.01)
        entry["p5_return"] = df_data["Returns"].quantile(0.05)
        entry["p10_return"] = df_data["Returns"].quantile(0.1)
        
        entry["p25_return_ann"] = annualize(df_data["Returns"].quantile(0.25), years)
        entry["p25_return"] = df_data["Returns"].quantile(0.25)
        entry["p50_return_ann"] = annualize(df_data["Returns"].quantile(0.5), years)
        entry["p50_return"] = df_data["Returns"].quantile(0.5)



        entry["mdds_mean"] = float(mdds_mean) * 100.0
        run_stats[m].append(entry)

display(Markdown("# Analysis Results"))
display(Markdown("## Simulation Parameters"))
display(Markdown(f"""| Initial NAV | Years to Simulate | Total Paths |
| :--- | :--- | :--- |
| ${initial_nav:,.2f} | {years} | {paths} |"""))

display(Markdown("## Results per model"))

for m in run_stats:
    data = run_stats[m]
    df_data = pd.DataFrame(data)
    display(Markdown(f"## {m}"))
    df_data = df_data.sort_values(by=['spending', 'p10_return'], ascending=[True, False])
    display(df_data)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(run_stats, f, indent=4)